# Superfícies e sólidos de revolução

**Módulo 06 — Geometria Espacial** · Notebook 17 de 18

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/leandrofreiredealmeida/curso_matematica_python/blob/main/06_geometria_espacial/0617a_superficies_e_solidos_de_revolucao.ipynb)

**Pré-requisitos**
- Módulo 06 — Geometria Espacial: [`0611a_cilindro.ipynb`](0611a_cilindro.ipynb), [`0612a_cone.ipynb`](0612a_cone.ipynb) e [`0613a_esfera.ipynb`](0613a_esfera.ipynb) (os três corpos redondos), [`0614a_solidos_semelhantes_e_troncos.ipynb`](0614a_solidos_semelhantes_e_troncos.ipynb) (tronco de cone) e [`0616a_inscricao_e_circunscricao_corpos_redondos.ipynb`](0616a_inscricao_e_circunscricao_corpos_redondos.ipynb) (a ideia de seção meridiana)
- Módulo 04 — Geometria Plana: [`0410a_pontos_notaveis_do_triangulo.ipynb`](../04_geometria_plana/0410a_pontos_notaveis_do_triangulo.ipynb) (baricentro) e [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (relações métricas no triângulo retângulo)

**Objetivos de aprendizagem** — ao final deste notebook você será capaz de:
- reconhecer superfícies e sólidos de revolução e identificar a figura que os gera;
- usar os Teoremas de Pappus-Guldin para calcular áreas e volumes de revolução;
- redescobrir as fórmulas do cilindro, do cone, da esfera e calcular as do toro;
- calcular volumes de sólidos de revolução quaisquer somando discos.

**Tempo estimado:** 70 a 90 minutos

### Antes de começar

Rode a célula abaixo antes de qualquer outra. Ela carrega o `utils_3d.py`, o arquivo com as funções de desenho deste módulo.

In [1]:
# Célula de preparação: deixa o notebook pronto para rodar no seu computador ou no Colab.
import sys

if "google.colab" in sys.modules:
    # No Colab, o notebook chega sozinho; baixamos o arquivo de desenhos do repositório.
    import urllib.request
    url = "https://raw.githubusercontent.com/leandrofreiredealmeida/curso_matematica_python/main/06_geometria_espacial/utils_3d.py"
    urllib.request.urlretrieve(url, "utils_3d.py")

import utils_3d as u3d
from ipywidgets import interact
import ipywidgets as widgets
import math
import numpy as np

## Quanto cabe neste vaso?

Um oleiro coloca a argila num torno, uma mesa que gira, e vai moldando a peça com as mãos. Tudo o que ele faz é escolher o **contorno** do vaso de um lado; o giro do torno faz o resto, repetindo esse contorno em volta toda. Torneiros de madeira e de metal trabalham do mesmo jeito: pés de mesa, peças de xadrez, maçanetas, garrafas e taças nascem assim.

O vaso abaixo tem um contorno curvo, que não é cilindro, nem cone, nem esfera. Não existe uma "fórmula do vaso". Mesmo assim, até o fim deste notebook você vai saber calcular quanto cabe dentro dele.

Neste notebook, toda figura que gira é descrita no **plano meridiano** (o plano que contém o eixo, como no notebook 16) com duas medidas: $r$ é a **distância até o eixo** e $z$ é a **altura**. Nos desenhos 3D, o eixo de rotação é sempre vertical.

🕹️ Gire o torno e veja o vaso nascer do seu contorno.

In [2]:
# Cores usadas em todo o notebook: cada papel tem a mesma cor na figura 3D e na
# figura plana (o gráfico 2D logo abaixo de cada figura 3D).
COR_FIGURA = u3d.CORES["solido"]               # a superfície ou o sólido que se forma
COR_GERATRIZ = u3d.CORES["destaque"]           # a curva ou a região que gira
COR_EIXO = u3d.CORES["apoio"]                  # o eixo de rotação
COR_CENTROIDE = u3d.CORES["plano_secundario"]  # o centroide e o caminho que ele faz


def criar_plano_meridiano(titulo: str):
    """Cria o gráfico 2D do plano meridiano: r na horizontal e z na vertical."""
    return u3d.criar_grafico(
        titulo=titulo, rotulo_x="r (distância ao eixo)", rotulo_y="z (altura)",
        proporcao_igual=True,
    )


def desenhar_eixo_no_plano(grafico, z_min: float, z_max: float):
    """Desenha o eixo de rotação (a reta r = 0) tracejado no plano meridiano."""
    folga = 0.15 * max(z_max - z_min, 1)
    return u3d.desenhar_curva(
        grafico, [0, 0], [z_min - folga, z_max + folga], rotulo="eixo de rotação",
        cor=COR_EIXO, tracejado=True,
    )


# O contorno do vaso no plano meridiano: para cada altura z (em cm), a distância r da
# parede até o eixo. O seno faz a parede "ondular": ela alarga, afina num pescoço e
# volta a abrir na boca.
alturas = np.linspace(0, 2 * math.pi, 80)
perfil_vaso = [(2 + 0.8 * math.sin(z), z) for z in alturas]   # (distância ao eixo, altura)


def girar_o_torno(giro: int) -> None:
    """Gira o contorno do vaso até o ângulo escolhido, em 3D e no plano meridiano."""
    fig = u3d.criar_figura(titulo=f"O torno girou {giro}°")
    fig = u3d.desenhar_superficie_de_revolucao(
        fig, perfil_vaso, angulo=giro, cor=COR_FIGURA, mostrar_paralelos=4
    )
    if giro == 360:
        # Volta completa: o vaso ganha o fundo, um disco com o raio do contorno em z = 0.
        raio_do_fundo = perfil_vaso[0][0]
        fig = u3d.desenhar_coroa_circular(
            fig, (0, 0, 0), raio_do_fundo, 0, cor=COR_FIGURA, opacidade=0.5
        )
    u3d.mostrar(fig)

    grafico = criar_plano_meridiano("O contorno escolhido pelo oleiro")
    grafico = u3d.desenhar_curva(
        grafico, [r for r, z in perfil_vaso], [z for r, z in perfil_vaso],
        rotulo="contorno (geratriz)", cor=COR_GERATRIZ,
    )
    grafico = desenhar_eixo_no_plano(grafico, 0, 2 * math.pi)
    u3d.mostrar(grafico)
    print("Quanto cabe neste vaso? Você vai descobrir na seção 5.")


interact(
    girar_o_torno,
    giro=widgets.IntSlider(
        min=15, max=360, step=15, value=90, description="giro (graus)",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(IntSlider(value=90, continuous_update=False, description='giro (graus)', max=360, min=15…

Toda figura que nasce de um contorno girando em torno de um eixo é uma figura **de revolução**. O cilindro, o cone e a esfera também são assim, e você vai ver que existe um único raciocínio que explica as fórmulas de todos eles.

## 1. Superfícies de revolução

> 📌 **Definição.** Uma **superfície de revolução** é a superfície gerada por uma curva plana (a **geratriz**) que gira em torno de uma reta do mesmo plano (o **eixo**), sem cruzá-la.

Durante o giro:
- cada ponto da geratriz descreve uma **circunferência**, perpendicular ao eixo e com centro nele: são os **paralelos** da superfície;
- cada posição da geratriz é um **meridiano**.

(É a mesma nomenclatura da esfera e da Terra, [notebook 13](0613a_esfera.ipynb)!)

| Geratriz | Superfície gerada |
|---|---|
| segmento paralelo ao eixo | superfície lateral de um **cilindro** |
| segmento com uma ponta no eixo | superfície lateral de um **cone** |
| segmento oblíquo, sem tocar o eixo | superfície lateral de um **tronco de cone** |
| semicircunferência com as pontas no eixo | **superfície esférica** |
| circunferência que não toca o eixo | **superfície tórica** (a de uma boia) |
| arco de parábola | **paraboloide** (antenas parabólicas, faróis) |

> 💡 **Dica.** As enormes torres de resfriamento das usinas, largas embaixo, estreitas no meio e abrindo de novo em cima, são superfícies de revolução chamadas **hiperboloides**. Curiosamente, elas podem ser construídas inteiramente com **vigas retas**, inclinadas e cruzadas.

🕹️ Escolha a geratriz e gire. As linhas claras na superfície são alguns paralelos.

In [3]:
# As geratrizes no plano meridiano: cada ponto é (r, z) = (distância ao eixo, altura).
meia_volta = np.linspace(0, math.pi, 50)       # ângulos para a semicircunferência
volta_inteira = np.linspace(0, 2 * math.pi, 80)  # ângulos para a circunferência
GERATRIZES = {
    "Segmento paralelo (cilindro)": [(2, 0), (2, 4)],
    "Segmento até o eixo (cone)": [(2, 0), (0, 4)],
    "Segmento oblíquo (tronco de cone)": [(3, 0), (1, 3)],
    # Semicircunferência de raio 2 e centro (0, 2): as pontas (0, 0) e (0, 4) ficam no eixo.
    "Semicircunferência (esfera)": [
        (2 * math.sin(t), 2 - 2 * math.cos(t)) for t in meia_volta
    ],
    # Circunferência de raio 1 e centro (3, 2): fica toda longe do eixo.
    "Circunferência afastada (toro)": [
        (3 + math.cos(t), 2 + math.sin(t)) for t in volta_inteira
    ],
    # Arco da parábola z = r²/2, de r = 0 até r = 2,5.
    "Arco de parábola (paraboloide)": [(r, r**2 / 2) for r in np.linspace(0, 2.5, 40)],
}


def girar_geratriz(geratriz: str, giro: int) -> None:
    """Gira a geratriz escolhida em torno do eixo e mostra o plano meridiano."""
    perfil = GERATRIZES[geratriz]
    fig = u3d.criar_figura(titulo=f"{geratriz}: giro de {giro}°")
    fig = u3d.desenhar_superficie_de_revolucao(
        fig, perfil, angulo=giro, cor=COR_FIGURA, mostrar_paralelos=3
    )
    u3d.mostrar(fig)

    rs = [r for r, z in perfil]
    zs = [z for r, z in perfil]
    grafico = criar_plano_meridiano("A geratriz no plano meridiano")
    grafico = u3d.desenhar_curva(grafico, rs, zs, rotulo="geratriz", cor=COR_GERATRIZ)
    grafico = desenhar_eixo_no_plano(grafico, min(zs), max(zs))
    u3d.mostrar(grafico)


interact(
    girar_geratriz,
    geratriz=widgets.Dropdown(
        options=list(GERATRIZES), description="geratriz",
        style={"description_width": "initial"},
    ),
    giro=widgets.IntSlider(
        min=15, max=360, step=15, value=270, description="giro (graus)",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(Dropdown(description='geratriz', options=('Segmento paralelo (cilindro)', 'Segmento até …

**Pare e pense**

1. Que curva, girando em torno de um eixo, gera a superfície de uma lata sem tampa e sem fundo?
2. Uma circunferência que **corta** o eixo pode ser a geratriz de uma superfície de revolução? Por quê?

<details>
<summary>Ver respostas</summary>

1. Um segmento paralelo ao eixo.
2. Não, pela nossa definição: a geratriz não pode cruzar o eixo. (Na prática, cada lado do eixo geraria uma superfície, e elas se sobreporiam.) Para a esfera, usamos só a **metade** da circunferência, com as pontas no eixo.
</details>

## 2. Sólidos de revolução

> 📌 **Definição.** Um **sólido de revolução** é o sólido gerado por uma **região plana** que gira em torno de uma reta do seu plano (o **eixo**), sem que a reta atravesse a região (ela pode tocar o contorno).

| Região que gira | Eixo | Sólido |
|---|---|---|
| retângulo | um dos lados | **cilindro** |
| triângulo retângulo | um dos catetos | **cone** |
| trapézio retângulo | o lado perpendicular às bases | **tronco de cone** |
| semicírculo | o diâmetro | **esfera** |
| círculo | uma reta afastada | **toro** (sólido) |
| retângulo | uma reta paralela a um lado, afastada | **cilindro com furo** (um tubo) |

> ⚠️ **Erro comum.** Achar que a mesma figura sempre gera o mesmo sólido. Um retângulo de lados 2 e 5, girando em torno do lado de 5, gera um cilindro de raio 2 e altura 5; girando em torno do lado de 2, gera um cilindro de raio 5 e altura 2. Os volumes são bem diferentes: $20\pi$ e $50\pi$!

**Um caso curioso.** Um triângulo retângulo girando em torno da **hipotenusa** gera **dois cones** unidos pela base (como um pião). O raio da base comum é a **altura relativa à hipotenusa** do triângulo.

🕹️ Escolha a região e o eixo. Num giro incompleto, as faces de corte (a região no começo e no fim do giro) mostram o "recheio" do sólido.

In [4]:
# O triângulo de catetos 3 e 4 girando em torno da hipotenusa: as relações métricas
# no triângulo retângulo (módulo 04) dão a posição do vértice do ângulo reto.
a, b = 3, 4
c = math.sqrt(a**2 + b**2)          # hipotenusa (fica sobre o eixo)
h = a * b / c                       # altura relativa à hipotenusa = raio da base comum
m = a**2 / c                        # projeção do cateto a sobre a hipotenusa

# As regiões no plano meridiano, com os vértices (r, z) na ordem do contorno.
semicirculo = [(2 * math.sin(t), -2 * math.cos(t)) for t in np.linspace(0, math.pi, 50)]
circulo = [
    (3 + math.cos(t), math.sin(t)) for t in np.linspace(0, 2 * math.pi, 60, endpoint=False)
]
REGIOES = {   # nome: (região, sólido gerado)
    "Retângulo em torno do lado maior": (
        [(0, 0), (2, 0), (2, 5), (0, 5)], "cilindro de raio 2 e altura 5"),
    "Retângulo em torno do lado menor": (
        [(0, 0), (5, 0), (5, 2), (0, 2)], "cilindro de raio 5 e altura 2"),
    "Triângulo retângulo em torno de um cateto": (
        [(0, 0), (3, 0), (0, 4)], "cone de raio 3 e altura 4"),
    "Trapézio retângulo": (
        [(0, 0), (3, 0), (1, 3), (0, 3)], "tronco de cone de raios 3 e 1 e altura 3"),
    "Semicírculo": (semicirculo, "esfera de raio 2"),
    "Círculo afastado (toro)": (
        circulo, "toro: tubo de raio 1, com o centro a 3 do eixo"),
    "Retângulo afastado (tubo)": (
        [(2, 0), (3, 0), (3, 4), (2, 4)],
        "cilindro de raio 3 e altura 4, com um furo de raio 2"),
    "Triângulo retângulo em torno da hipotenusa": (
        [(0, 0), (h, m), (0, c)],
        f"dois cones de raio {h:.1f}, com alturas {m:.1f} e {c - m:.1f}"),
}


def girar_regiao(solido: str, giro: int) -> None:
    """Gira a região escolhida em torno do eixo e mostra o plano meridiano."""
    regiao, descricao = REGIOES[solido]
    fig = u3d.criar_figura(titulo=f"Giro de {giro}°")
    fig = u3d.desenhar_solido_de_revolucao(
        fig, regiao, angulo=giro, cor=COR_FIGURA, opacidade=0.6
    )
    u3d.mostrar(fig)

    zs = [z for r, z in regiao]
    grafico = criar_plano_meridiano("A região no plano meridiano")
    grafico = u3d.desenhar_poligono_2d(grafico, regiao, rotulo="região", cor=COR_GERATRIZ)
    grafico = desenhar_eixo_no_plano(grafico, min(zs), max(zs))
    if solido == "Triângulo retângulo em torno da hipotenusa":
        # A altura relativa à hipotenusa: vira o raio da base comum dos dois cones.
        grafico = u3d.desenhar_curva(
            grafico, [0, h], [m, m], rotulo="altura h (raio comum)", cor=COR_FIGURA,
            tracejado=True,
        )
    u3d.mostrar(grafico)
    print(f"Sólido gerado: {descricao}.")


interact(
    girar_regiao,
    solido=widgets.Dropdown(
        options=list(REGIOES), description="região e eixo",
        style={"description_width": "initial"}, layout=widgets.Layout(width="420px"),
    ),
    giro=widgets.IntSlider(
        min=15, max=360, step=15, value=270, description="giro (graus)",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(Dropdown(description='região e eixo', layout=Layout(width='420px'), options=('Retângulo …

**Pare e pense**

1. Que região gera um tronco de cone? Em torno de que eixo?
2. Um triângulo retângulo de catetos 3 e 4 gira em torno da hipotenusa. Qual é o raio da "base comum" dos dois cones?

<details>
<summary>Ver respostas</summary>

1. Um trapézio retângulo, girando em torno do lado perpendicular às bases.
2. É a altura relativa à hipotenusa: $\frac{3 \cdot 4}{5} = 2{,}4$.
</details>

## 3. Os Teoremas de Pappus-Guldin

### O centroide

O **centroide** (ou **centro de gravidade**) de uma figura é o ponto onde ela ficaria em equilíbrio, se fosse recortada numa chapa fina e uniforme. Alguns centroides fáceis:
- de um **segmento**: o ponto médio;
- de um **retângulo** ou de um **círculo**: o centro;
- de um **triângulo**: o **baricentro**, o encontro das medianas, que fica a $\frac{1}{3}$ da altura, a partir de cada lado ([módulo 04](../04_geometria_plana/0410a_pontos_notaveis_do_triangulo.ipynb)).

### Os teoremas

Quando uma figura gira em torno do eixo, o seu centroide descreve uma circunferência de raio $d$ (a distância do centroide ao eixo), com comprimento $2\pi d$.

**Teorema 1 (área)**
> A área da superfície gerada por uma curva plana de comprimento $L$, girando em torno de um eixo que não a atravessa, é
> $$\boxed{S = L \cdot 2\pi d}$$
> em que $d$ é a distância do centroide da **curva** ao eixo.

**Teorema 2 (volume)**
> O volume do sólido gerado por uma região plana de área $A$, girando em torno de um eixo que não a atravessa, é
> $$\boxed{\text{volume} = A \cdot 2\pi d}$$
> em que $d$ é a distância do centroide da **região** ao eixo.

Em palavras: **o tamanho da figura vezes o caminho percorrido pelo seu centro**.

> 🕵️ **Curiosidade histórica.** **Papo de Alexandria** (século IV) enunciou esses resultados na sua *Coleção Matemática*, uma das últimas grandes obras da matemática grega antiga. Mais de mil anos depois, o matemático suíço **Paul Guldin** (1577–1643) os redescobriu e divulgou. Por isso eles carregam os dois nomes.

> ⚠️ **Erro comum.** Usar a distância do eixo até um **lado** ou um **vértice** da figura, em vez da distância até o **centroide**. Cada ponto da figura percorre um caminho diferente; o centroide representa a "média" desses caminhos.

🕹️ Escolha uma figura e a distância do eixo, e compare o resultado de Pappus com a fórmula conhecida. O controle `largura` vale para o retângulo e o triângulo; o controle `raio`, para o círculo. O `afastamento` é a distância do eixo até a figura (a parte dela mais perto do eixo).

In [5]:
def aplicar_pappus(
    figura: str, largura: float, altura: float, raio: float, afastamento: float
) -> None:
    """Calcula o volume por Pappus, confere com a fórmula conhecida e desenha."""
    w, h = largura, altura
    if figura == "Retângulo (cilindro)":
        # Retângulo de largura w (na direção de r) e altura h, a partir de r = afastamento.
        regiao = [(afastamento, 0), (afastamento + w, 0), (afastamento + w, h), (afastamento, h)]
        area = w * h
        d = afastamento + w / 2                  # o centroide é o centro do retângulo
        z_centroide = h / 2
        volume_conferencia = math.pi * ((afastamento + w)**2 - afastamento**2) * h
        formula = "cilindro de raio externo menos o furo, π(R² − r²)h"
    elif figura == "Triângulo retângulo (cone)":
        # Cateto w sobre a base, cateto h junto ao eixo (ângulo reto em r = afastamento).
        regiao = [(afastamento, 0), (afastamento + w, 0), (afastamento, h)]
        area = w * h / 2
        d = afastamento + w / 3                  # baricentro: a 1/3 do cateto vertical
        z_centroide = h / 3
        # Com afastamento 0, é o cone. Afastado, é um tronco de cone (raios
        # afastamento + w e afastamento) menos o cilindro do furo (raio afastamento).
        R, r = afastamento + w, afastamento
        volume_conferencia = math.pi * h / 3 * (R**2 + R * r + r**2) - math.pi * r**2 * h
        formula = "cone" if afastamento == 0 else "tronco de cone menos o furo"
    else:
        # Círculo de raio a, com o centro a distância b do eixo.
        a = raio
        b = afastamento + a                      # o eixo fica a "afastamento" da borda
        regiao = [
            (b + a * math.cos(t), a * math.sin(t))
            for t in np.linspace(0, 2 * math.pi, 80, endpoint=False)
        ]
        area = math.pi * a**2
        d = b                                    # o centroide é o centro do círculo
        z_centroide = 0
        volume_conferencia = None                # fórmula nova: o toro (seção 4)

    caminho = 2 * math.pi * d                    # a circunferência descrita pelo centroide
    volume_pappus = area * caminho

    # Figura 3D: giro de 300°, para ver o "corte". O centroide e o seu caminho em verde.
    fig = u3d.criar_figura(titulo="Volume = área × caminho do centroide")
    fig = u3d.desenhar_solido_de_revolucao(
        fig, regiao, angulo=300, cor=COR_FIGURA, opacidade=0.35
    )
    fig = u3d.desenhar_circunferencia(
        fig, (0, 0, z_centroide), d, cor=COR_CENTROIDE, espessura=5
    )
    centroide_3d = (d * math.cos(math.radians(300)), d * math.sin(math.radians(300)), z_centroide)
    fig = u3d.desenhar_ponto(fig, centroide_3d, rotulo="centroide", cor=COR_CENTROIDE)
    u3d.mostrar(fig)

    # Figura 2D: a região, o eixo, o centroide e a distância d.
    zs = [z for r, z in regiao]
    grafico = criar_plano_meridiano("A região que gira e o seu centroide")
    grafico = u3d.desenhar_poligono_2d(grafico, regiao, rotulo="região", cor=COR_GERATRIZ)
    grafico = desenhar_eixo_no_plano(grafico, min(zs), max(zs))
    grafico = u3d.desenhar_curva(
        grafico, [0, d], [z_centroide, z_centroide], rotulo="distância d",
        cor=COR_CENTROIDE, tracejado=True,
    )
    grafico = u3d.marcar_ponto_no_grafico(
        grafico, d, z_centroide, rotulo="centroide", cor=COR_CENTROIDE
    )
    u3d.mostrar(grafico)

    print(f"Área da região: A = {area:.4f}")
    print(f"Distância do centroide ao eixo: d = {d:.4f}")
    print(f"Caminho do centroide: 2πd = {caminho:.4f}")
    print(f"Volume por Pappus: A · 2πd = {volume_pappus:.4f}")
    if volume_conferencia is None:
        print("Conferência: este é o toro, cuja fórmula você vai ver na seção 4.")
    else:
        print(f"Conferência ({formula}): {volume_conferencia:.4f}")
    area_medida, centroide_medido = u3d.medir_area_e_centroide_2d(regiao)
    print(f"Conferência com o utils_3d: área {area_medida:.4f}, "
          f"centroide a {centroide_medido[0]:.4f} do eixo")


interact(
    aplicar_pappus,
    figura=widgets.Dropdown(
        options=["Retângulo (cilindro)", "Triângulo retângulo (cone)", "Círculo (toro)"],
        description="figura", style={"description_width": "initial"},
    ),
    largura=widgets.FloatSlider(
        min=0.5, max=3, step=0.25, value=2, description="largura",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    altura=widgets.FloatSlider(
        min=0.5, max=3, step=0.25, value=3, description="altura",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    raio=widgets.FloatSlider(
        min=0.5, max=3, step=0.25, value=1, description="raio",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    afastamento=widgets.FloatSlider(
        min=0, max=3, step=0.25, value=0, description="afastamento",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(Dropdown(description='figura', options=('Retângulo (cilindro)', 'Triângulo retângulo (co…

**Pare e pense**

1. Um retângulo de lados 2 e 5 gira em torno do lado de 5. Use Pappus para calcular o volume.
2. Por que, no triângulo do cone, o centroide fica a $\frac{r}{3}$ do eixo, e não a $\frac{r}{2}$?

<details>
<summary>Ver respostas</summary>

1. Área $= 10$; o centro está a $1$ do eixo; volume $= 10 \cdot 2\pi \cdot 1 = 20\pi$.
2. O centroide do triângulo é o baricentro, que fica a $\frac{1}{3}$ da distância entre um lado e o vértice oposto. O triângulo é "mais gordo" perto do eixo e afina para longe dele, então o centroide fica mais perto do eixo.
</details>

## 4. Redescobrindo as fórmulas

Com Pappus, as fórmulas dos notebooks anteriores saem em uma linha:

| Sólido | Figura que gira | Pappus | Resultado |
|---|---|---|---|
| cilindro (volume) | retângulo $r \times h$ | $rh \cdot 2\pi \cdot \frac{r}{2}$ | $\pi r^2 h$ |
| cilindro (área lateral) | segmento $h$ a distância $r$ | $h \cdot 2\pi r$ | $2\pi r h$ |
| cone (volume) | triângulo de catetos $r$ e $h$ | $\frac{rh}{2} \cdot 2\pi \cdot \frac{r}{3}$ | $\frac{1}{3}\pi r^2 h$ |
| cone (área lateral) | segmento $g$ com centro a $\frac{r}{2}$ | $g \cdot 2\pi \cdot \frac{r}{2}$ | $\pi r g$ |
| tronco de cone (área lateral) | segmento $g$ com centro a $\frac{R + r}{2}$ | $g \cdot 2\pi \cdot \frac{R + r}{2}$ | $\pi(R + r)g$ |

### O toro: uma fórmula nova

O **toro** é gerado por um círculo de raio $a$ girando em torno de um eixo a uma distância $b$ do centro do círculo ($b > a$). É o formato de uma boia, de um donut, de uma câmara de pneu.

$$\text{volume} = \pi a^2 \cdot 2\pi b = \boxed{2\pi^2 a^2 b} \qquad\qquad S = 2\pi a \cdot 2\pi b = \boxed{4\pi^2 a b}$$

Repare: é como se o toro fosse um **cilindro dobrado**, com base de área $\pi a^2$ (ou contorno $2\pi a$) e "altura" $2\pi b$, o comprimento do caminho do centro.

### A esfera ao contrário: descobrindo centroides

Para a esfera, Pappus funciona "de trás para a frente": já conhecemos o volume e a área ([notebook 13](0613a_esfera.ipynb)), e podemos **descobrir** os centroides!
- **Semicírculo** de raio $R$ (região): $\frac{4}{3}\pi R^3 = \frac{\pi R^2}{2} \cdot 2\pi d$, então $d = \dfrac{4R}{3\pi} \approx 0{,}42R$.
- **Semicircunferência** de raio $R$ (curva): $4\pi R^2 = \pi R \cdot 2\pi d$, então $d = \dfrac{2R}{\pi} \approx 0{,}64R$.

🕹️ Monte um toro.

In [6]:
def montar_toro(raio_do_tubo: float, distancia_ao_eixo: float, giro: int) -> None:
    """Desenha o toro (inteiro ou cortado) e calcula o volume e a área por Pappus."""
    a, b = raio_do_tubo, distancia_ao_eixo
    if b <= a:
        print("Para formar um toro, o círculo não pode atravessar o eixo:")
        print("escolha a distância ao eixo b maior que o raio do tubo a.")
        return

    # O círculo de raio a e centro (b, 0) no plano meridiano, aproximado por um
    # polígono regular de 80 lados. A função devolve pontos (x, y, z) no plano z = 0;
    # usamos x como r (distância ao eixo) e y como z (altura).
    circulo = [(x, y) for x, y, _ in u3d.poligono_regular(80, raio=a, centro=(b, 0, 0))]

    fig = u3d.criar_figura(titulo=f"Toro com a = {a:g} e b = {b:g}")
    fig = u3d.desenhar_solido_de_revolucao(
        fig, circulo, angulo=giro, cor=COR_FIGURA, opacidade=0.6
    )
    # O caminho do centro do círculo: uma circunferência de raio b.
    fig = u3d.desenhar_circunferencia(fig, (0, 0, 0), b, cor=COR_CENTROIDE, tracejado=True)
    u3d.mostrar(fig)

    grafico = criar_plano_meridiano("O círculo que gira")
    grafico = u3d.desenhar_circunferencia_2d(
        grafico, (b, 0), a, rotulo="círculo de raio a", cor=COR_GERATRIZ, preencher=True
    )
    grafico = desenhar_eixo_no_plano(grafico, -a, a)
    grafico = u3d.desenhar_curva(
        grafico, [0, b], [0, 0], rotulo="distância b", cor=COR_CENTROIDE, tracejado=True
    )
    grafico = u3d.marcar_ponto_no_grafico(grafico, b, 0, rotulo="centro", cor=COR_CENTROIDE)
    u3d.mostrar(grafico)

    volume = 2 * math.pi**2 * a**2 * b
    area = 4 * math.pi**2 * a * b
    volume_em_litros = volume / 1000   # se as medidas estiverem em cm
    print(f"Volume = 2π²a²b = {volume:.4f}  (em cm³, são {volume_em_litros:.4f} litros)")
    print(f"Área   = 4π²ab = {area:.4f}")


interact(
    montar_toro,
    raio_do_tubo=widgets.FloatSlider(
        min=0.5, max=2, step=0.25, value=1, description="raio do tubo a",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    distancia_ao_eixo=widgets.FloatSlider(
        min=1, max=5, step=0.25, value=3, description="distância ao eixo b",
        style={"description_width": "initial"}, continuous_update=False,
    ),
    giro=widgets.IntSlider(
        min=30, max=360, step=30, value=360, description="giro (graus)",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

# Uma boia de piscina: tubo de raio 15 cm, centro do tubo a 45 cm do centro da boia.
volume_boia = 2 * math.pi**2 * 15**2 * 45   # em cm³
print(f"Uma boia com a = 15 cm e b = 45 cm guarda cerca de {volume_boia / 1000:.0f} "
      "litros de ar.")

interactive(children=(FloatSlider(value=1.0, continuous_update=False, description='raio do tubo a', max=2.0, m…

Uma boia com a = 15 cm e b = 45 cm guarda cerca de 200 litros de ar.


**Pare e pense**

1. Calcule o volume e a área de um toro com $a = 1$ e $b = 3$.
2. Qual é a distância do centroide de um semicírculo de raio 3 até o diâmetro?

<details>
<summary>Ver respostas</summary>

1. Volume $= 2\pi^2 \cdot 1 \cdot 3 = 6\pi^2 \approx 59{,}2$. Área $= 4\pi^2 \cdot 3 = 12\pi^2 \approx 118{,}4$.
2. $d = \frac{4 \cdot 3}{3\pi} = \frac{4}{\pi} \approx 1{,}27$.
</details>

## 5. Discos empilhados: o volume de qualquer sólido de revolução

Pappus precisa do centroide, que nem sempre é fácil de achar. Mas existe outro caminho, que funciona para **qualquer** contorno.

Corte o sólido de revolução em fatias finas, perpendiculares ao eixo. Cada fatia é quase um **cilindro baixinho** (um disco): se a fatia está na altura $z$ e tem espessura $\Delta z$, o seu raio é $r(z)$ (a distância do contorno ao eixo nessa altura), e o seu volume é aproximadamente
$$\pi \, r(z)^2 \, \Delta z$$

Somando todas as fatias:
$$\text{volume} \approx \sum \pi \, r(z_i)^2 \, \Delta z$$

Quanto mais fatias, melhor a aproximação. É a mesma ideia das pilhas de cartas do [notebook 09](0609a_volume_e_principio_de_cavalieri.ipynb) e das placas da pirâmide do [notebook 10](0610a_piramide.ipynb), agora com discos.

> 📎 **Conexão.** No módulo 11 (Cálculo), você vai ver o que acontece quando o número de discos "vai para o infinito" e a espessura "vai para zero": a soma vira um valor exato, com um nome e um símbolo próprios. É exatamente assim que os engenheiros calculam volumes de tanques, garrafas e peças torneadas.

🕹️ Confira o método com sólidos conhecidos e depois responda à pergunta do gancho. Na figura plana, cada disco aparece como um retângulo: a metade da seção meridiana do disco.

In [7]:
def raio_esfera(z):        # esfera de raio 2, com z de -2 a 2
    return math.sqrt(max(0, 4 - z**2))


def raio_cone(z):          # cone de raio 2 e altura 4, com z de 0 a 4 (vértice em cima)
    return 2 * (1 - z / 4)


def raio_vaso(z):          # o vaso do gancho, com z de 0 a 2π
    return 2 + 0.8 * math.sin(z)


SOLIDOS = {   # nome: (função do raio, z inicial, z final, volume exato conhecido)
    "Esfera de raio 2": (raio_esfera, -2, 2, 32 * math.pi / 3),
    "Cone de raio 2 e altura 4": (raio_cone, 0, 4, 16 * math.pi / 3),
    "O vaso do gancho": (raio_vaso, 0, 2 * math.pi, None),   # não há fórmula pronta
}


def somar_discos(raio, z_inicial: float, z_final: float, numero_de_discos: int):
    """Soma os volumes dos discos que aproximam o sólido (raio de cada um no meio
    da fatia). Devolve a soma e as alturas dos meios das fatias."""
    dz = (z_final - z_inicial) / numero_de_discos
    soma = 0
    alturas_dos_discos = []
    for i in range(numero_de_discos):
        z_meio = z_inicial + (i + 0.5) * dz
        soma = soma + math.pi * raio(z_meio)**2 * dz
        alturas_dos_discos.append(z_meio)
    return soma, alturas_dos_discos


def empilhar_discos(solido: str, numero_de_discos: int) -> None:
    """Desenha a pilha de discos dentro do sólido e compara a soma com o exato."""
    raio, z_inicial, z_final, exato = SOLIDOS[solido]
    soma, alturas_dos_discos = somar_discos(raio, z_inicial, z_final, numero_de_discos)
    dz = (z_final - z_inicial) / numero_de_discos

    # Figura 3D: o sólido quase transparente e, dentro dele, os discos (cada disco é
    # um polígono regular de 40 lados, que imita o círculo, com espessura dz).
    perfil = [(raio(z), z) for z in np.linspace(z_inicial, z_final, 100)]
    fig = u3d.criar_figura(titulo=f"{solido}: {numero_de_discos} discos")
    fig = u3d.desenhar_superficie_de_revolucao(
        fig, perfil, cor=COR_FIGURA, opacidade=0.15, mostrar_perfil=False
    )
    discos = [
        u3d.poligono_regular(40, raio=raio(z_meio), centro=(0, 0, z_meio - dz / 2))
        for z_meio in alturas_dos_discos
    ]
    fig = u3d.desenhar_pilha(fig, discos, espessura=dz)
    u3d.mostrar(fig)

    # Figura 2D: o contorno e, para cada disco, o retângulo de largura r(z) e altura dz.
    grafico = criar_plano_meridiano("O contorno e os discos no plano meridiano")
    for z_meio in alturas_dos_discos:
        baixo, cima, r = z_meio - dz / 2, z_meio + dz / 2, raio(z_meio)
        grafico = u3d.desenhar_poligono_2d(
            grafico, [(0, baixo), (r, baixo), (r, cima), (0, cima)], cor=COR_FIGURA,
            opacidade=0.4, espessura=1,
        )
    grafico = u3d.desenhar_curva(
        grafico, [r for r, z in perfil], [z for r, z in perfil], rotulo="contorno",
        cor=COR_GERATRIZ,
    )
    grafico = desenhar_eixo_no_plano(grafico, z_inicial, z_final)
    u3d.mostrar(grafico)

    print(f"Soma de {numero_de_discos} discos: {soma:.4f}")
    if exato is not None:
        print(f"Volume exato: {exato:.4f}   erro: {soma - exato:+.4f} "
              f"({100 * (soma - exato) / exato:+.2f}%)")
    else:
        volume_vaso, _ = somar_discos(raio_vaso, 0, 2 * math.pi, 1000)
        print(f"Com 1000 discos, o volume do vaso é {volume_vaso:.2f} cm³.")

    # A tabela: o que acontece com a soma quando o número de discos aumenta.
    print()
    print("     N |       soma")
    for n in [2, 5, 10, 50, 100, 1000]:
        soma_n, _ = somar_discos(raio, z_inicial, z_final, n)
        print(f"{n:6d} | {soma_n:10.4f}")


interact(
    empilhar_discos,
    solido=widgets.Dropdown(
        options=list(SOLIDOS), description="sólido",
        style={"description_width": "initial"},
    ),
    numero_de_discos=widgets.IntSlider(
        min=2, max=60, step=2, value=10, description="número de discos",
        style={"description_width": "initial"}, continuous_update=False,
    ),
);

interactive(children=(Dropdown(description='sólido', options=('Esfera de raio 2', 'Cone de raio 2 e altura 4',…

Com 1000 discos, o vaso do gancho tem cerca de **85 cm³**. (Com 10 000 discos, o valor quase não muda: a soma já "estabilizou".) Não existe uma fórmula de vaso, mas existe um **método** que funciona para todos.

**Pare e pense**

1. Na tabela da esfera, a soma dos discos ficou acima ou abaixo do valor exato? O erro diminui quando o número de discos aumenta?
2. Por que o método dos discos funciona para sólidos de revolução, mas não diretamente para uma pirâmide de base quadrada?

<details>
<summary>Ver respostas</summary>

1. Com os pontos médios, a soma fica um pouco acima do valor exato da esfera, e o erro diminui rapidamente com mais discos (com 10 discos, o erro já é menor que 1%).
2. Porque as fatias de uma pirâmide de base quadrada são **quadrados**, não discos. O método funciona, mas com "placas quadradas" no lugar de discos, como no notebook 10.
</details>

## Resumo

**Superfície de revolução:** curva plana (geratriz) girando em torno de um eixo do seu plano, sem cruzá-lo. Pontos da geratriz descrevem **paralelos**; posições da geratriz são **meridianos**.

**Sólido de revolução:** região plana girando em torno de um eixo do seu plano.
Retângulo → cilindro; triângulo retângulo → cone; trapézio retângulo → tronco de cone; semicírculo → esfera; círculo afastado → toro.

**Centroides:** segmento → ponto médio; retângulo e círculo → centro; triângulo → baricentro ($\frac{1}{3}$ da altura); semicírculo → $\frac{4R}{3\pi}$; semicircunferência → $\frac{2R}{\pi}$.

**Pappus-Guldin:** $S = L \cdot 2\pi d$ (curva); volume $= A \cdot 2\pi d$ (região), com $d$ = distância do **centroide** ao eixo.

**Toro** (raio do tubo $a$, distância do centro ao eixo $b$): volume $= 2\pi^2 a^2 b$; $S = 4\pi^2 ab$.

**Discos empilhados:** volume $\approx \sum \pi\, r(z_i)^2 \,\Delta z$; funciona para qualquer contorno e melhora com mais discos.

## Exercícios

Responda preenchendo as variáveis indicadas em cada célula (troque cada `...` pela sua resposta) e rode a célula de verificação logo abaixo. Os exercícios estão em três níveis: **básico**, **intermediário** e **desafio**. Use `math.pi` nos cálculos.

### Básico

In [ ]:
# Exercício Básico 1: verdadeiro ou falso? Troque cada ... por True ou False.
# (a) Um retângulo girando em torno de um dos lados gera um cilindro.
# (b) Um triângulo retângulo girando em torno de um cateto gera um cone.
# (c) Um semicírculo girando em torno do diâmetro gera uma esfera.
# (d) Pelo Teorema de Pappus, o volume é a área da região vezes o comprimento da
#     circunferência descrita pelo seu centroide.
# (e) Um retângulo de lados 2 e 5 gera o mesmo cilindro, qualquer que seja o lado
#     escolhido como eixo.
respostas_ex1 = {
    "a": ...,
    "b": ...,
    "c": ...,
    "d": ...,
    "e": ...,
}

print(respostas_ex1)

In [ ]:
# Verificação
gabarito_ex1 = {"a": True, "b": True, "c": True, "d": True, "e": False}
dicas_ex1 = {
    "a": "o lado que fica no eixo vira a altura do cilindro (seção 2 - Sólidos de revolução)",
    "b": "o cateto que fica no eixo vira a altura do cone (seção 2 - Sólidos de revolução)",
    "c": "o diâmetro fica no eixo e o arco gera a superfície esférica (seção 2 - Sólidos de revolução)",
    "d": "volume = A · 2πd, e 2πd é o caminho do centroide (seção 3 - Os Teoremas de Pappus-Guldin)",
    "e": "troque o lado do eixo e compare os raios e as alturas (seção 2 - Sólidos de revolução)",
}
for item, certa in gabarito_ex1.items():
    assert respostas_ex1[item] == certa, (
        f"Tente de novo, revise: o item ({item}) não está certo; {dicas_ex1[item]}."
    )
print("Certo!")

In [ ]:
# Exercício Básico 2: o retângulo 2 × 5. Calcule o volume do cilindro gerado pelo
# retângulo de lados 2 e 5 girando:
# (a) em torno do lado de 5;
# (b) em torno do lado de 2.
volume_lado_maior_ex2 = ...
volume_lado_menor_ex2 = ...

print(volume_lado_maior_ex2, volume_lado_menor_ex2)

In [ ]:
# Verificação
respostas_ex2 = [
    ("a", volume_lado_maior_ex2, 20 * math.pi),
    ("b", volume_lado_menor_ex2, 50 * math.pi),
]
for item, resposta, certa in respostas_ex2:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), o lado que fica no eixo é a altura do "
        "cilindro; o outro é o raio (seção 2 - Sólidos de revolução)."
    )
print("Certo!")

In [ ]:
# Exercício Básico 3: o toro. Um círculo de raio a = 1 gira em torno de um eixo a
# uma distância b = 3 do seu centro. Calcule:
# (a) o volume do toro;
# (b) a área da superfície do toro.
volume_toro_ex3 = ...
area_toro_ex3 = ...

print(volume_toro_ex3, area_toro_ex3)

In [ ]:
# Verificação
respostas_ex3 = [
    ("a", volume_toro_ex3, 6 * math.pi**2),
    ("b", area_toro_ex3, 12 * math.pi**2),
]
for item, resposta, certa in respostas_ex3:
    assert resposta is not ... and abs(resposta - certa) <= 0.01, (
        f"Tente de novo, revise: no item ({item}), volume = 2π²a²b e área = 4π²ab "
        "(seção 4 - Redescobrindo as fórmulas)."
    )
print("Certo!")

### Intermediário

In [ ]:
# Exercício Intermediário 4: o pião. Um triângulo retângulo de catetos 3 e 4 gira
# em torno da hipotenusa. Calcule o volume do sólido:
# (a) somando os volumes dos dois cones;
# (b) pelo Teorema de Pappus.
# Os dois valores devem coincidir!
volume_cones_ex4 = ...
volume_pappus_ex4 = ...

print(volume_cones_ex4, volume_pappus_ex4)

In [ ]:
# Verificação
assert volume_cones_ex4 is not ... and abs(volume_cones_ex4 - 9.6 * math.pi) <= 0.01, (
    "Tente de novo, revise: no item (a), o raio comum é a altura relativa à hipotenusa "
    "(2,4) e as alturas dos dois cones somam 5 (seção 2 - Sólidos de revolução)."
)
assert volume_pappus_ex4 is not ... and abs(volume_pappus_ex4 - 9.6 * math.pi) <= 0.01, (
    "Tente de novo, revise: no item (b), a área é 6 e o baricentro fica a 1/3 de 2,4 do "
    "eixo (seção 3 - Os Teoremas de Pappus-Guldin)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 5: a boia. Uma boia de piscina tem a forma de um toro, com
# tubo de raio 15 cm e distância de 45 cm do centro do tubo até o centro da boia.
# Quantos litros de ar cabem nela? (1 litro = 1000 cm³)
litros_ex5 = ...

print(litros_ex5)

In [ ]:
# Verificação
assert litros_ex5 is not ... and abs(litros_ex5 - 2 * math.pi**2 * 225 * 45 / 1000) <= 0.5, (
    "Tente de novo, revise: calcule o volume em cm³ com 2π²a²b e divida por 1000 "
    "(seção 4 - Redescobrindo as fórmulas)."
)
print("Certo!")

In [ ]:
# Exercício Intermediário 6: o centroide do semicírculo. Use o Teorema de Pappus "de
# trás para a frente", com o volume da esfera, para calcular a distância do centroide
# de um semicírculo de raio 3 até o diâmetro.
d_ex6 = ...

print(d_ex6)

In [ ]:
# Verificação
assert d_ex6 is not ... and abs(d_ex6 - 4 / math.pi) <= 0.01, (
    "Tente de novo, revise: (4/3)πR³ = (πR²/2) · 2π · d; isole d "
    "(seção 4 - Redescobrindo as fórmulas)."
)
print("Certo!")

### Desafio

In [ ]:
# Exercício Desafio 7: quantos discos? Para a esfera de raio 1 (z de -1 a 1), a soma
# dos discos com N fatias (pelo ponto médio) é sempre um pouco maior que o volume
# exato 4π/3. Encontre o MENOR N para o qual o erro relativo, (soma − exato) / exato,
# fica abaixo de 1% (0,01).
exato = 4 * math.pi / 3
N = 0
erro_relativo = 1   # começa grande, só para entrar no laço
while erro_relativo >= 0.01 and N < 1000:   # o limite de 1000 evita um laço sem fim
    N = N + 1
    dz = 2 / N
    soma = 0
    for i in range(N):
        ...  # complete aqui: a altura do meio da fatia i e a soma do volume do disco
    ...  # complete aqui: calcule erro_relativo

menor_N_ex7 = ...

print(menor_N_ex7)

In [ ]:
# Verificação
assert menor_N_ex7 is not ... and menor_N_ex7 == 8, (
    "Tente de novo, revise: use um laço while; o raio do disco na altura z é √(1 − z²) "
    "e o erro relativo é (soma − exato) / exato (seção 5 - Discos empilhados)."
)
print("Certo!")

In [ ]:
# Exercício Desafio 8: outro vaso. Um vaso tem o contorno r(z) = 2 + sen(z), com z de
# 0 a 2π (medidas em cm). Calcule o volume com 1000 discos.
numero_de_discos = 1000
z_inicial, z_final = 0, 2 * math.pi
dz = (z_final - z_inicial) / numero_de_discos
soma = 0
for i in range(numero_de_discos):
    ...  # complete aqui: a altura do meio da fatia i
    ...  # complete aqui: some o volume do disco, π r² dz, com r = 2 + sen(z)

volume_ex8 = ...

print(volume_ex8)

In [ ]:
# Verificação
assert volume_ex8 is not ... and abs(volume_ex8 - 9 * math.pi**2) <= 0.1, (
    "Tente de novo, revise: adapte o laço da seção 5 trocando a função do raio por "
    "2 + math.sin(z) (seção 5 - Discos empilhados)."
)
print("Esse valor pode ser obtido de forma exata no Cálculo (módulo 11): é 9π² ≈ 88,83.")
print("O seu código chegou lá sem nenhuma fórmula de vaso!")
print("Certo!")

## Conexões

**Isso depende de:**
- [`0609a_volume_e_principio_de_cavalieri.ipynb`](0609a_volume_e_principio_de_cavalieri.ipynb) e [`0610a_piramide.ipynb`](0610a_piramide.ipynb) (somas de fatias: as pilhas de cartas e as placas da pirâmide)
- [`0611a_cilindro.ipynb`](0611a_cilindro.ipynb), [`0612a_cone.ipynb`](0612a_cone.ipynb), [`0613a_esfera.ipynb`](0613a_esfera.ipynb) e [`0614a_solidos_semelhantes_e_troncos.ipynb`](0614a_solidos_semelhantes_e_troncos.ipynb) (as fórmulas redescobertas por Pappus)
- [`0616a_inscricao_e_circunscricao_corpos_redondos.ipynb`](0616a_inscricao_e_circunscricao_corpos_redondos.ipynb) (a seção meridiana, que aqui vira a figura que gira)
- [`0410a_pontos_notaveis_do_triangulo.ipynb`](../04_geometria_plana/0410a_pontos_notaveis_do_triangulo.ipynb) (baricentro) e [`0416a_triangulos_retangulos.ipynb`](../04_geometria_plana/0416a_triangulos_retangulos.ipynb) (relações métricas no triângulo retângulo)

**Isso será usado em:**
- 📌 Superfícies e sólidos esféricos (zonas, calotas, segmentos e setores esféricos por Pappus e por discos)
- 📌 Geometria Analítica (parábolas e paraboloides)
- 📌 Limite, Integral e Derivada (volumes de revolução)